# Frozen-backbone tekne eğitimini (`boat_v4s_frozen`) Colab GPU'da devam ettir

Mac'te epoch **42/80**'de duran eğitimi, aynı checkpoint'ten (optimizer durumu dahil) GPU'da kaldığı yerden devam ettirir. Gerçek `resume=True` — epoch sayacı, öğrenme oranı programı ve optimizer momentumu korunur.

**Önce:**
1. `Çalışma zamanı > Çalışma zamanı türünü değiştir` ile bir GPU seç (A100/L4/T4).
2. `boat_v4s_frozen_resume.zip` dosyasını (Mac'indeki `depth-anything` klasöründe, ~386MB) Google Drive'ına yükle.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Zip'i Drive'da nereye yüklediysen ZIP_PATH'i ona göre ayarla
# (bir önceki denemede zip'ler 'trainFreeze' klasörüne gitmişti, oraya baktım)
ZIP_PATH = '/content/drive/MyDrive/trainFreeze/boat_v4s_frozen_resume.zip'

!rm -rf /content/work
!mkdir -p /content/work
!unzip -q "$ZIP_PATH" -d /content/work
!find /content/work -maxdepth 5 -iname 'last.pt' -o -iname 'dataset.yaml'

In [ ]:
!pip install -q ultralytics

In [ ]:
# dataset.yaml içindeki Mac path'ini Colab path'ine çevir
import pathlib

yaml_path = pathlib.Path('/content/work/yolo_dataset_v4/dataset.yaml')
content = yaml_path.read_text()
new_content = content.replace(
    '/Users/armin/Desktop/depth-anything/yolo_dataset_v4',
    '/content/work/yolo_dataset_v4'
)
yaml_path.write_text(new_content)
print(yaml_path.read_text())

In [ ]:
# KRITIK ADIM: checkpoint'in icine gomulu train_args'i Colab'a gore duzelt.
# Mac'te 'device: mps' ve save_dir Mac'in mutlak yoluydu -- bunlari degistirmezsek
# ultralytics /Users/armin/... altina yazmaya calisir ve/veya cihazi bulamaz.
import torch, glob

ckpt_path = glob.glob('/content/work/**/boat_v4s_frozen/weights/last.pt', recursive=True)[0]
print('checkpoint:', ckpt_path)
run_dir = str(pathlib.Path(ckpt_path).parent.parent)
print('run_dir:', run_dir)

ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
ta = ckpt['train_args']
print('--- eski ---')
print({k: ta[k] for k in ('device', 'data', 'save_dir', 'project', 'name')})

ta['device'] = 0
ta['data'] = '/content/work/yolo_dataset_v4/dataset.yaml'
ta['save_dir'] = run_dir
ta['workers'] = 2  # Colab'da cok cekirdek yok, 8 kalirsa yavaslatabilir
ckpt['train_args'] = ta
torch.save(ckpt, ckpt_path)

# args.yaml dosyasini da ayni sekilde guncelle (bazi ultralytics surumleri oradan da okuyor)
import yaml
args_yaml_path = pathlib.Path(run_dir) / 'args.yaml'
with open(args_yaml_path) as f:
    y = yaml.safe_load(f)
y['device'] = 0
y['data'] = '/content/work/yolo_dataset_v4/dataset.yaml'
y['save_dir'] = run_dir
y['workers'] = 2
with open(args_yaml_path, 'w') as f:
    yaml.safe_dump(y, f)

print('--- yeni ---')
print({k: ta[k] for k in ('device', 'data', 'save_dir', 'project', 'name')})

In [ ]:
# Eğitimi epoch 42'den (43'ten) devam ettir -- gercek resume, optimizer durumu korunur
from ultralytics import YOLO

model = YOLO(ckpt_path)
results = model.train(resume=True)

In [ ]:
# Eğitim Colab oturumu koptuğu için durduysa (12 saat limiti, boşta kalma vb.),
# yukarıdaki hücreleri (Drive bağlama, unzip -- ZIP_PATH'i Drive'daki yeni konuma göre
# ayarlayarak son results'ları içeren bir zip yükleyip -- ve bu hücreyi) tekrar çalıştır.
# Not: last.pt her epoch sonunda güncellendiği için, koptuğu yerden otomatik devam eder.

In [ ]:
# Bitince (veya ara sonuçları kaybetmemek için ara sıra) Drive'a kopyala
!mkdir -p /content/drive/MyDrive/boat_v4s_frozen_results
!cp -r "$run_dir" /content/drive/MyDrive/boat_v4s_frozen_results/
print('Kopyalandı: Google Drive > boat_v4s_frozen_results')

## Bitince Mac'e geri alma

Drive > `boat_v4s_frozen_results` altındaki klasörü indir, içindeki `weights/best.pt` ve `weights/last.pt`'yi Mac'inde `depth-anything/runs/detect/runs_boat_yolo/boat_v4s_frozen/weights/` altına koy (üzerine yaz).